# 🩸 Blood Glucose Forecaster - Advanced Training Pipeline

**Model**: XGBoost Regressor with GPU acceleration  
**Target**: Predict blood glucose 30 minutes into the future  
**Features**: CGM lags, deltas, time encodings, demographics

## Cell 1: Setup

In [ ]:
# Install required packages
!pip install -q polars xgboost matplotlib joblib

# Imports
import polars as pl
import xgboost as xgb
import numpy as np
import matplotlib.pyplot as plt
import joblib
import json
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✅ Setup complete!")
print(f"Polars version: {pl.__version__}")
print(f"XGBoost version: {xgb.__version__}")

## Cell 2: Data Loading & Initial Filter

In [ ]:
# Define data path - UPDATE THIS PATH TO YOUR DATA LOCATION
DATA_PATH = "/content/drive/MyDrive/Diabete/data/metabonet_public_train.parquet"

# Load raw parquet
print("📂 Loading raw data...")
df = pl.read_parquet(DATA_PATH)
print(f"Raw data shape: {df.shape}")
print(f"Columns: {df.columns}")

# Select required columns
required_cols = ['id', 'date', 'CGM', 'basal', 'bolus', 'carbs', 'age', 'weight']

# Check which columns exist
available_cols = [col for col in required_cols if col in df.columns]
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    print(f"⚠️ Missing columns (will be filled): {missing_cols}")

# Select available columns
df = df.select(available_cols)

# Add missing columns with default values
for col in missing_cols:
    if col in ['age', 'weight']:
        df = df.with_columns(pl.lit(-1.0).alias(col))
    elif col in ['basal', 'bolus', 'carbs']:
        df = df.with_columns(pl.lit(0.0).alias(col))

print(f"\n📊 After column selection: {df.shape}")

# Handle nulls
print("\n🔧 Handling null values...")
print(f"Null counts before:\n{df.null_count()}")

# CGM: Drop rows where null
df = df.filter(pl.col("CGM").is_not_null())

# basal, bolus, carbs: Fill nulls with 0.0
df = df.with_columns([
    pl.col("basal").fill_null(0.0),
    pl.col("bolus").fill_null(0.0),
    pl.col("carbs").fill_null(0.0),
])

# age, weight: Fill nulls with -1.0 (standard imputation for demographics)
df = df.with_columns([
    pl.col("age").fill_null(-1.0),
    pl.col("weight").fill_null(-1.0),
])

print(f"\n✅ After null handling: {df.shape}")
print(f"Null counts after:\n{df.null_count()}")
print(f"\n📋 Sample data:\n{df.head(5)}")

## Cell 3: Feature Engineering (Advanced)

In [ ]:
print("🔧 Starting Advanced Feature Engineering...")

# VITAL: Sort by patient and date
df = df.sort(["id", "date"])
print("✅ Sorted by ['id', 'date']")

# Ensure date is datetime
if df["date"].dtype != pl.Datetime:
    df = df.with_columns(pl.col("date").cast(pl.Datetime))

# === TARGET (y) ===
# Create future_glucose = CGM shifted by -6 (30 mins ahead, assuming 5-min intervals)
df = df.with_columns(
    pl.col("CGM").shift(-6).over("id").alias("future_glucose")
)
print("✅ Created target: future_glucose (30 min ahead)")

# === TIME FEATURES ===
df = df.with_columns([
    pl.col("date").dt.hour().alias("hour"),
    pl.col("date").dt.weekday().alias("weekday"),  # 0=Monday, 6=Sunday
])

# Cyclical encoding for hour
df = df.with_columns([
    (2 * np.pi * pl.col("hour") / 24).sin().alias("hour_sin"),
    (2 * np.pi * pl.col("hour") / 24).cos().alias("hour_cos"),
])

# is_weekend: 1 if Saturday(5) or Sunday(6), else 0
df = df.with_columns(
    pl.when(pl.col("weekday") >= 5).then(1).otherwise(0).alias("is_weekend")
)
print("✅ Created time features: hour_sin, hour_cos, is_weekend")

# === TREND FEATURES (Crucial Additions) ===
# cgm_delta_5min: Current CGM - CGM(t-5min) = CGM - shift(1)
df = df.with_columns(
    (pl.col("CGM") - pl.col("CGM").shift(1).over("id")).alias("cgm_delta_5min")
)

# cgm_delta_15min: Current CGM - CGM(t-15min) = CGM - shift(3)
df = df.with_columns(
    (pl.col("CGM") - pl.col("CGM").shift(3).over("id")).alias("cgm_delta_15min")
)
print("✅ Created trend features: cgm_delta_5min, cgm_delta_15min")

# === LAG FEATURES ===
df = df.with_columns([
    pl.col("CGM").shift(1).over("id").alias("lag_1"),   # 5 min ago
    pl.col("CGM").shift(2).over("id").alias("lag_2"),   # 10 min ago
    pl.col("CGM").shift(3).over("id").alias("lag_3"),   # 15 min ago
    pl.col("CGM").shift(6).over("id").alias("lag_6"),   # 30 min ago
])
print("✅ Created lag features: lag_1, lag_2, lag_3, lag_6")

# === CLEANUP ===
# Drop rows with NaN created by shifting/lagging/differencing
print(f"\n📊 Before cleanup: {df.shape}")
df = df.drop_nulls()
print(f"📊 After cleanup: {df.shape}")

# Display feature info
print(f"\n📋 Final columns: {df.columns}")
print(f"\n📋 Sample engineered data:\n{df.head(5)}")

## Cell 4: Train/Test Split (Patient-Block Split)

In [ ]:
print("🔀 Performing Patient-Block Train/Test Split...")
print("(Using last 20% of each patient's data as test set to prevent data leakage)")

# Add row number within each patient
df = df.with_columns(
    pl.col("id").cum_count().over("id").alias("row_num")
)

# Calculate total rows per patient
df = df.with_columns(
    pl.col("id").count().over("id").alias("total_rows")
)

# Calculate split threshold (80% of data for training)
df = df.with_columns(
    (pl.col("total_rows") * 0.8).floor().alias("split_threshold")
)

# Create train/test flag: 1 if row_num <= 80% of total, else 0
df = df.with_columns(
    pl.when(pl.col("row_num") <= pl.col("split_threshold"))
    .then(pl.lit("train"))
    .otherwise(pl.lit("test"))
    .alias("split")
)

# Split into train and test sets
train_df = df.filter(pl.col("split") == "train")
test_df = df.filter(pl.col("split") == "test")

print(f"\n📊 Train set: {train_df.shape}")
print(f"📊 Test set: {test_df.shape}")
print(f"📊 Train ratio: {len(train_df) / len(df):.2%}")

# Define feature columns (exclude metadata and target)
exclude_cols = ['id', 'date', 'future_glucose', 'row_num', 'total_rows', 'split_threshold', 'split', 'hour', 'weekday']
feature_cols = [col for col in df.columns if col not in exclude_cols]

print(f"\n🎯 Feature columns ({len(feature_cols)}): {feature_cols}")

# Prepare training and test data
X_train = train_df.select(feature_cols).to_numpy()
y_train = train_df.select("future_glucose").to_numpy().ravel()

X_test = test_df.select(feature_cols).to_numpy()
y_test = test_df.select("future_glucose").to_numpy().ravel()

print(f"\n✅ X_train shape: {X_train.shape}")
print(f"✅ y_train shape: {y_train.shape}")
print(f"✅ X_test shape: {X_test.shape}")
print(f"✅ y_test shape: {y_test.shape}")

## Cell 5: Model Training

In [ ]:
print("🚀 Training XGBoost Regressor with GPU acceleration...")

# Model configuration
model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    device='cuda',  # GPU acceleration
    random_state=42,
    early_stopping_rounds=50,
    verbosity=1
)

print("\n📋 Model Configuration:")
print(f"  - Objective: reg:squarederror")
print(f"  - n_estimators: 1000")
print(f"  - learning_rate: 0.05")
print(f"  - max_depth: 6")
print(f"  - tree_method: hist (GPU)")
print(f"  - device: cuda")
print(f"  - early_stopping_rounds: 50")

# Train with early stopping using eval set
print("\n⏳ Training started...")
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=100  # Print every 100 rounds
)

print(f"\n✅ Training complete!")
print(f"Best iteration: {model.best_iteration}")
print(f"Best score: {model.best_score:.4f}")

## Cell 6: Evaluation

In [ ]:
print("📊 Evaluating Model Performance...")

# Make predictions
y_pred = model.predict(X_test)

# Calculate metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print("\n" + "="*50)
print("🎯 MODEL PERFORMANCE METRICS")
print("="*50)
print(f"  RMSE: {rmse:.4f} mg/dL")
print(f"  MAE:  {mae:.4f} mg/dL")
print("="*50)

# Calculate additional statistics
errors = y_test - y_pred
print(f"\n📈 Error Statistics:")
print(f"  Mean Error: {np.mean(errors):.4f}")
print(f"  Std Error:  {np.std(errors):.4f}")
print(f"  Min Error:  {np.min(errors):.4f}")
print(f"  Max Error:  {np.max(errors):.4f}")

# Plot: Actual vs Predicted for 200 time steps
plt.figure(figsize=(14, 6))

n_samples = 200
time_steps = range(n_samples)

plt.plot(time_steps, y_test[:n_samples], label='Actual', color='#2ecc71', linewidth=2, alpha=0.8)
plt.plot(time_steps, y_pred[:n_samples], label='Predicted', color='#e74c3c', linewidth=2, alpha=0.8, linestyle='--')

plt.xlabel('Time Steps (5-min intervals)', fontsize=12)
plt.ylabel('Blood Glucose (mg/dL)', fontsize=12)
plt.title('🩸 Blood Glucose Forecaster: Actual vs Predicted (30-min ahead)', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Add metrics annotation
textstr = f'RMSE: {rmse:.2f}\nMAE: {mae:.2f}'
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
plt.gca().text(0.02, 0.98, textstr, transform=plt.gca().transAxes, fontsize=10,
        verticalalignment='top', bbox=props)

plt.show()

# Feature importance plot
print("\n📊 Feature Importance:")
importance_df = pl.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort('importance', descending=True)

print(importance_df)

# Plot feature importance
plt.figure(figsize=(10, 6))
plt.barh(importance_df['feature'].to_list()[::-1], importance_df['importance'].to_list()[::-1], color='#3498db')
plt.xlabel('Importance', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.title('🎯 Feature Importance', fontsize=14)
plt.tight_layout()
plt.show()

## Cell 7: Save Artifacts

In [ ]:
print("💾 Saving Model Artifacts...")

# Define output path - UPDATE THIS PATH AS NEEDED
OUTPUT_DIR = "/content/drive/MyDrive/Diabete/models/"

# Create directory if it doesn't exist
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save model as JSON
model_path = OUTPUT_DIR + "glucose_forecaster_advanced_v1.json"
model.save_model(model_path)
print(f"✅ Model saved: {model_path}")

# CRUCIAL: Save feature names for API
features_path = OUTPUT_DIR + "forecaster_features.json"
feature_info = {
    "feature_names": feature_cols,
    "n_features": len(feature_cols),
    "target": "future_glucose",
    "prediction_horizon": "30 minutes (6 steps of 5-min intervals)",
    "metrics": {
        "rmse": float(rmse),
        "mae": float(mae)
    },
    "model_params": {
        "n_estimators": 1000,
        "learning_rate": 0.05,
        "max_depth": 6,
        "best_iteration": int(model.best_iteration) if hasattr(model, 'best_iteration') else None
    }
}

with open(features_path, 'w') as f:
    json.dump(feature_info, f, indent=2)
print(f"✅ Feature info saved: {features_path}")

# Display saved feature info
print("\n📋 Saved Feature Information:")
print(json.dumps(feature_info, indent=2))

print("\n" + "="*50)
print("🎉 PIPELINE COMPLETE!")
print("="*50)
print(f"\n📁 Saved artifacts:")
print(f"   1. Model: {model_path}")
print(f"   2. Features: {features_path}")